In [ ]:
from imports.models import *
from imports.utils import *
import adabound as Adabound
import itertools
import numpy as np

In [ ]:
from multiprocessing import Pool
POOL_PROCESS = 10

In [ ]:
A_SIDE_NUM = 3
A_SIDE_RESOL = 600
A_TARGET_MARGIN = 25
A_SIDE_MARGIN = 300
A_FAR_SIDE_MARGIN = 5000
A_HIER_MARGIN = 25
A_HIER_MARGIN = 25
SIDE_CANDI_NUM = 2
B_TARGET_GAP = 0

In [ ]:
CUDA_DEVICE = 0
N_PULSE = 32
IMAGE_WIDTH = 10
TIME_RANGE_32  = 7000
TIME_RANGE_256  = 0
EXISTING_SPINS = 0
EVALUATION_ALL = 0

target_side_distance = 3000
A_init  = 10000
A_final = 20000
A_step  = 200
A_range = 200
B_init  = 1500
B_final = 50000
noise_scale = 0.5
zero_scale = 0.05
SAVE_DIR_NAME = "./test_data/"
is_CNN = 0
is_remove_model_index = 0

In [ ]:
PRE_PROCESS = False
PRE_SCALE = 1
MAGNETIC_FIELD = 403.553                        # # The external magnetic field strength. Unit: Gauss
GYRO_MAGNETIC_RATIO = 1.0705*1000               # Unit: Herts
WL_VALUE = MAGNETIC_FIELD*GYRO_MAGNETIC_RATIO*2*np.pi

In [ ]:
EXP_DATA_32_PATH = '../data/exp_data/exp_data_32.npy'                        # the experimental data to be evalutated
EXP_DATA_DENO_32_PATH = '../data/exp_data/exp_data_32_deno.npy'              # the denoised experimental data to be evalutated
EXP_DATA_256_PATH = '../data/exp_data/exp_data_256.npy'                      # the experimental data to be evalutated
EXP_DATA_DENO_256_PATH = '../data/exp_data/exp_data_256_deno.npy'            # the denoised experimental data to be evalutated
TIME_DATA_32_PATH = '../data/time_data/time_data_32.npy'                     # the time data for the experimental data to be evalutated
TIME_DATA_256_PATH = '../data/time_data/time_data_256.npy'                   # the time data for the experimental data to be evalutated
SPIN_BATH_32_PATH = '../data/spin_bath/spin_bath_M_value_N32.npy'            # the spin bath data for the experimental N_PULSE (it is not pre-requisite so one can just ignore this line.)
SPIN_BATH_256_PATH = '../data/spin_bath/spin_bath_M_value_N256.npy'          # the spin bath data for the experimental N_PULSE (it is not pre-requisite so one can just ignore this line.)
TOTAL_INDICES_32_PATH = '../data/total_indices/total_indices_v4_N32.npy'     # pre-calculated time_indexing file by using Eqn.(4) in the maintext
TOTAL_INDICES_256_PATH = '../data/total_indices/total_indices_v4_N256.npy'

In [ ]:
exp_data_32 = np.load(EXP_DATA_32_PATH)
exp_data_deno_32 = np.load(EXP_DATA_DENO_32_PATH)
exp_data_256 = np.load(EXP_DATA_256_PATH)
exp_data_deno_256 = np.load(EXP_DATA_DENO_256_PATH)
time_data_32 = np.load(TIME_DATA_32_PATH)
time_data_256 = np.load(TIME_DATA_256_PATH)
spin_bath_32 = np.load(SPIN_BATH_32_PATH)
spin_bath_256 = np.load(SPIN_BATH_256_PATH)
total_indices_32 = np.load(TOTAL_INDICES_32_PATH, allow_pickle=True).item()
total_indices_256 = np.load(TOTAL_INDICES_256_PATH, allow_pickle=True).item()

In [ ]:
# 첫 번재 파일 로드하여 초기 AB_lists_dic  todtjd
AB_lists_dic = np.load('./data/AB_target_dic/AB_target_dic_v4_s0.npy', allow_pickle=True).item() # a pre-calculated grouped list of nuclear spins with the same target period.
# 나머지 파일들을 반복문으로 로드하여  update
for i in range(1, 16):
    temp = np.load(f'./data/AB_target_dic/AB_target_dic_v4_s{i}.npy', allow_pickle=True).item()
    AB_lists_dic.update(temp)

In [ ]:
def estimate_specific_AB_values(predicted_periods):

    model_lists = np.array([[A_temp_list[0], A_temp_list[-1], B_init, B_final] for A_temp_list in predicted_periods])

    total_raw_pred_list = []
    total_deno_pred_list = []
    total_A_lists = []
    total_results = []
    print("==========================================================================================")
    print('image_width:{}, TIME_RANGE_32:{}, TIME_RANGE_256:{}, B_init:{}, B_end:{}'.format(IMAGE_WIDTH, TIME_RANGE_32, TIME_RANGE_256, B_init, B_final))
    print("==========================================================================================")

    for model_idx, [A_first, A_end, B_first, B_end] in enumerate(model_lists):
        print("========================================================================")
        print('A_first:{}, A_end:{}, B_first:{}, B_end:{}'.format(A_first, A_end, B_first, B_end))
        print("========================================================================")
        A_num = 1
        B_num = 1
        A_resol, B_resol = 50, B_end-B_first

        A_idx_list = np.arange(A_first, A_end+A_resol, A_num*A_resol)
        if (B_end-B_first)%B_resol==0:
            B_idx_list = np.arange(B_first, B_first+B_resol, B_num*B_resol)
        else:
            B_idx_list = np.arange(B_first, B_end, B_num*B_resol)
        AB_idx_set = [[A_idx, B_idx] for A_idx, B_idx in itertools.product(A_idx_list, B_idx_list)]

        A_side_num = A_SIDE_NUM
        A_side_resol = A_SIDE_RESOL
        A_target_margin = A_TARGET_MARGIN
        A_side_margin = A_SIDE_MARGIN
        A_far_side_margin = A_FAR_SIDE_MARGIN
        A_hier_margin = A_HIER_MARGIN
        side_candi_num = SIDE_CANDI_NUM        # the number of "how many times" to generate 'AB_side_candidate'

        if N_PULSE==32:
            B_side_min, B_side_max = 6000, 70000
            B_side_gap = 5000
            B_target_gap = 1000  # distance between targets only valid when B_num >= 2.
            distance_btw_target_side = A_target_margin+A_side_margin+target_side_distance # the final distance between target and side = distance_btw_target_side - (A_target_margin+A_side_margin)

        elif N_PULSE==256:
            B_side_min, B_side_max = 1000, 20000
            B_side_gap = 0      # distance between target and side (applied for both side_same and side)
            B_target_gap = 0
            distance_btw_target_side = A_target_margin+A_side_margin+target_side_distance

        PRE_PROCESS, PRE_SCALE = False, 1
        if ((N_PULSE == 32) & (B_first<12000)):
            PRE_PROCESS = True
            PRE_SCALE = 8
            print("==================== PRE_PROCESSING:True =====================")

        class_num = A_num*B_num + 1
        cpu_num_for_multi = 10
        batch_for_multi = 256
        class_batch = cpu_num_for_multi*batch_for_multi

        spin_zero_scale = {'same':zero_scale, 'side':0.50, 'mid':0.05, 'far':0.05}  # setting 'same'=1.0 for hierarchical model

        epochs = 20
        valid_batch = 4096
        valid_mini_batch = 1024
        num_of_summation = 4

        args = (AB_lists_dic, N_PULSE, A_num, B_num, A_resol, B_resol, A_side_num, A_side_resol, B_side_min,
                    B_side_max, B_target_gap, B_side_gap, A_target_margin, A_side_margin, A_far_side_margin,
                    class_batch, class_num, spin_zero_scale, distance_btw_target_side, side_candi_num)

        for class_idx in range(num_of_summation):
            TPk_AB_candi, _, temp_hier_target_AB_candi  = gen_TPk_AB_candidates(AB_idx_set, True, *args)
            temp_hier_target_AB_candi[:,:,0] = get_marginal_arr(temp_hier_target_AB_candi[:,:,0], A_hier_margin)
            if class_idx==0:
                total_hier_target_AB_candi = temp_hier_target_AB_candi[:]
                target_candidates = TPk_AB_candi[1, :, 0, :]
                side_candidates   = TPk_AB_candi[0, :, 0, :]
                rest_candidates   = TPk_AB_candi[1, :, 1:, :]
            else:
                total_hier_target_AB_candi = np.concatenate((total_hier_target_AB_candi, temp_hier_target_AB_candi[:]), axis=1)
                target_candidates = np.concatenate((target_candidates, TPk_AB_candi[1, :, 0, :]), axis=0)
                side_candidates   = np.concatenate((side_candidates, TPk_AB_candi[0, :, 0, :]), axis=0)
                rest_candidates   = np.concatenate((rest_candidates, TPk_AB_candi[1, :, 1:, :]), axis=0)

        hier_indices = return_total_hier_index_list(A_idx_list, cut_threshold=4)
        print(f"hier_indices: {hier_indices}")
        if len(hier_indices)==0 or len(hier_indices[-1])==0 or len(hier_indices[-1][0])==0:
            print("⚠️ hier_indices가 비어 있거나 구조적으로 잘못되었습니다. total_class_num = 1")
        total_class_num = len(hier_indices[-1][0]) + 1

        total_TPk_AB_candidates = np.zeros((total_class_num, num_of_summation*TPk_AB_candi.shape[1], total_class_num+TPk_AB_candi.shape[2]+2, 2))
        indices = np.random.randint(rest_candidates.shape[0], size=(total_class_num, rest_candidates.shape[0]))
        total_TPk_AB_candidates[:, :, (total_class_num-1):-4, :] = rest_candidates[indices]
        indices = np.random.randint(side_candidates.shape[0], size=(total_class_num, side_candidates.shape[0], 2))
        total_TPk_AB_candidates[:, :, -4:-2, :] = side_candidates[indices]
        indices = np.random.randint(side_candidates.shape[0], size=(total_class_num, side_candidates.shape[0], 2))
        total_TPk_AB_candidates[:, :, -2:, :] = side_candidates[indices]

        # if EXISTING_SPINS:
        #     total_TPk_AB_candidates = return_existing_spins_wrt_margins(deno_pred_N32_B12000_above, total_TPk_AB_candidates, A_existing_margin, B_existing_margin)
        final_TPk_AB_candidates = total_TPk_AB_candidates[:1]
        for class_idx, hier_index in enumerate(hier_indices):
            temp_batch = total_TPk_AB_candidates.shape[1] // len(hier_index)
            for idx2, index in enumerate(hier_index):
                temp = np.swapaxes(total_hier_target_AB_candi[index], 0, 1)
                if idx2 < (len(hier_index)-1):
                    temp_idx = np.random.randint(total_hier_target_AB_candi.shape[1], size=(temp_batch))
                    total_TPk_AB_candidates[class_idx+1, (idx2)*temp_batch:(idx2+1)*temp_batch, :len(index)] = temp[temp_idx]
                else:
                    residual_batch = total_TPk_AB_candidates[class_idx+1, (idx2)*temp_batch:, :len(index)].shape[0]
                    temp_idx = np.random.randint(total_hier_target_AB_candi.shape[1], size=(residual_batch))
                    total_TPk_AB_candidates[class_idx+1, (idx2)*temp_batch:, :len(index)] = temp[temp_idx]
            final_TPk_AB_candidates = np.concatenate((final_TPk_AB_candidates, total_TPk_AB_candidates[class_idx+1:class_idx+2, :, :]), axis=0)
        median_A_idx = len(AB_idx_set) // 2

        # below is for generating N32 datasets
        print("Generating N32 data..")
        model_index_32 = get_model_index(total_indices_32, AB_idx_set[median_A_idx][0], time_thres_idx=TIME_RANGE_32, image_width=IMAGE_WIDTH)
        if (AB_idx_set[median_A_idx][0] > 13000) | (AB_idx_set[median_A_idx][0] < -13000):
            model_index_32, _ = return_index_without_A_idx(total_indices_32, model_index_32, 0, TIME_RANGE_32, 5)
        total_class_num = final_TPk_AB_candidates.shape[0]
        X_train_arr = np.zeros((total_class_num, final_TPk_AB_candidates.shape[1], model_index_32.shape[0], 2*IMAGE_WIDTH+1))
        Y_train_arr = np.zeros((total_class_num, final_TPk_AB_candidates.shape[1], total_class_num))
        mini_batch = X_train_arr.shape[1] // cpu_num_for_multi
        for class_idx in range(total_class_num):
            Y_train_arr[class_idx, :, class_idx] = 1
            for idx1 in range(cpu_num_for_multi):
                AB_lists_batch = final_TPk_AB_candidates[class_idx, idx1*mini_batch:(idx1+1)*mini_batch]
                globals()["pool_{}".format(idx1)] = Pool.apply_async(gen_M_arr_batch, [AB_lists_batch, model_index_32, time_data_32[:TIME_RANGE_32],
                                                                                        WL_VALUE, 32, PRE_PROCESS, PRE_SCALE,
                                                                                        noise_scale, spin_bath_32[:TIME_RANGE_32]])

            for idx2 in range(cpu_num_for_multi):
                X_train_arr[class_idx, idx2*mini_batch:(idx2+1)*mini_batch] = globals()["pool_{}".format(idx2)].get(timeout=None)
            print("class_idx:", class_idx, end=' ')

        X_train_arr = X_train_arr.reshape(-1, model_index_32.flatten().shape[0])
        Y_train_arr = Y_train_arr.reshape(-1, Y_train_arr.shape[2])

        # below is for generating N256 datasets
        # if TIME_RANGE_256:
        #     print("Generating N256 data..")
        #     model_index_256 = get_model_index(total_indices_256, AB_idx_set[median_A_idx][0], time_thres_idx=TIME_RANGE_256, image_width=IMAGE_WIDTH)
        #     if (AB_idx_set[median_A_idx][0] > 13000) | (AB_idx_set[median_A_idx][0] < -13000):
        #         model_index_256, _ = return_index_without_A_idx(total_indices_256, model_index_256, 0, TIME_RANGE_256, 5)
        #     total_class_num = final_TPk_AB_candidates.shape[0]
        #     X_train_256 = np.zeros((total_class_num, final_TPk_AB_candidates.shape[1], model_index_256.shape[0], 2*IMAGE_WIDTH+1))
        #     Y_train_256 = np.zeros((total_class_num, final_TPk_AB_candidates.shape[1], total_class_num))
        #     mini_batch = X_train_256.shape[1] // cpu_num_for_multi
        #     for class_idx in range(total_class_num):
        #         Y_train_256[class_idx, :, class_idx] = 1
        #         for idx1 in range(cpu_num_for_multi):
        #             AB_lists_batch = final_TPk_AB_candidates[class_idx, idx1*mini_batch:(idx1+1)*mini_batch]
        #             globals()["pool_{}".format(idx1)] = Pool.apply_async(gen_M_arr_batch, [AB_lists_batch, model_index_256, time_data_256[:TIME_RANGE_256],
        #                                                                                     WL_VALUE, 256, PRE_PROCESS, PRE_SCALE,
        #                                                                                     noise_scale, spin_bath_256[:TIME_RANGE_256]])
        #
        #         for idx2 in range(cpu_num_for_multi):
        #             X_train_256[class_idx, idx2*mini_batch:(idx2+1)*mini_batch] = globals()["pool_{}".format(idx2)].get(timeout=None)
        #         print("class_idx:", class_idx, end=' ')
        #
        #     X_train_256 = X_train_256.reshape(-1, model_index_256.flatten().shape[0])
        #     Y_train_256 = Y_train_256.reshape(-1, Y_train_256.shape[2])
        #     X_train_arr = np.concatenate((X_train_32, X_train_256), axis=1)

        model = HPC(X_train_arr.shape[-1], Y_train_arr.shape[-1]).cuda()
        try:
            model(torch.Tensor(X_train_arr[:16]).cuda())
        except:
            raise NameError("The input shape should be revised")

        total_parameter = sum(p.numel() for p in model.parameters())
        print('total_parameter: ', total_parameter / 1000000, 'M')

        MODEL_PATH = './data/models/'
        mini_batch_list = [2048]
        learning_rate_list = [1e-6]
        op_list = [['Adabound', [1.0, 0.95, 0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6, 0.55, 0.5]]]
        criterion = nn.BCELoss().cuda()

        hyperparameter_set = [[mini_batch, learning_rate, selected_optim_name] for mini_batch, learning_rate, selected_optim_name in itertools.product(mini_batch_list, learning_rate_list, op_list)]
        print("==================== A_idx: {}, B_idx: {} ======================".format(A_first, B_first))

        total_loss, total_val_loss, total_acc, trained_model = train(MODEL_PATH, N_PULSE, X_train_arr, Y_train_arr, model, hyperparameter_set, criterion,
                                                                    epochs, valid_batch, valid_mini_batch, exp_data_32, is_pred=False, is_print_results=False,
                                                                    is_preprocess=PRE_PROCESS, PRE_SCALE=PRE_SCALE, model_index=model_index_32,
                                                                    exp_data_deno=exp_data_deno_32, is_regression=False)

        model.load_state_dict(torch.load(trained_model[0][0]))
        model.eval()
        print("Model loaded as evalutation mode. Model path:", trained_model[0][0])

        if TIME_RANGE_256:
            exp_data_test = np.hstack((exp_data_256[model_index_256.flatten()], exp_data_32[model_index_32.flatten()]))
        else:
            exp_data_test = exp_data_32[model_index_32.flatten()]
        exp_data_test = 1-(2*exp_data_test - 1)
        exp_data_test = exp_data_test.reshape(1, -1)
        exp_data_test = torch.Tensor(exp_data_test).cuda()

        pred = model(exp_data_test)
        pred = pred.detach().cpu().numpy()
        total_raw_pred_list.append([pred[0], total_class_num, hier_indices[-1][0].__len__()])
        print("raw", np.argmax(pred), np.max(pred), pred)

        if TIME_RANGE_256:
            exp_data_test = np.hstack((exp_data_deno_256[model_index_256.flatten()], exp_data_deno_32[model_index_32.flatten()]))
        else:
            exp_data_test = exp_data_deno_32[model_index_32.flatten()]
        exp_data_test = 1-(2*exp_data_test - 1)
        exp_data_test = exp_data_test.reshape(1, -1)
        exp_data_test = torch.Tensor(exp_data_test).cuda()

        pred = model(exp_data_test)
        pred = pred.detach().cpu().numpy()
        total_deno_pred_list.append([pred[0], total_class_num, hier_indices[-1][0].__len__()])
        print("deno", np.argmax(pred), np.max(pred), pred)
        print("Config:N_PULSE{}, B_init{}, B_final{}, w{}_batch{}, zero{}".format(N_PULSE, B_init, B_final, IMAGE_WIDTH, batch_for_multi, zero_scale))
        total_results.append([total_raw_pred_list[model_idx], total_deno_pred_list[model_idx], hier_indices])

    return total_results

In [ ]:
def split_A_range(A_init, A_final, step=50, group_size = 20):
    A_values = list(range(A_init, A_final, step))
    grouped = [A_values[i:i+group_size] for i in range(0, len(A_values), group_size)]
    return grouped
predicted_periods = split_A_range(A_init, A_final)

In [ ]:
# 디버깅
print(f"predicted_periods: {predicted_periods}")
print(f"len(predicted_periods): {len(predicted_periods)}")

In [ ]:
regression_results = estimate_specific_AB_values(predicted_periods)

In [ ]:
# B 값이 15000 이상인 하이퍼핀 파라미터만 필터링해서 저장
filtered_results = [res for res in regression_results if res[1] > 15000]  # res = [A, B] 형태라고 가정
filtered_results = np.array(filtered_results)

save_path = SAVE_DIR_NAME + 'predicted_results_N32_B15000above.npy'
np.save(save_path, filtered_results)
print(f"✅ B > 15000인 결과를 {save_path} 에 저장했습니다.")